### Initialization

In [1]:
agent_pids = []

### Add Agent

In [2]:
from pathlib import Path
import subprocess

parent_dir = Path.cwd().parent
proc_agent = subprocess.Popen(["node", "main.js"], cwd=str(parent_dir))
print(f"Running (PID={proc_agent.pid})"); agent_pids.append(proc_agent.pid)

Running (PID=6684)


In [3]:
import time
time.sleep(15)

In [4]:
# # If necessary, when something goes wrong during debugging, kill proc with pid
# import subprocess
# for pid in [49676]: #agent_pids:
#     print(subprocess.run(["taskkill", "/PID", str(pid), "/F"]))

### Construction and Evaluation

In [5]:
# --- Load prompts from the mapping (single source of truth) ---
from pathlib import Path
import json

# Load all JSON files from benchmarks folder
BENCHMARKS_DIR = Path.cwd() / "benchmarks"
mapping = []
for json_file in sorted(BENCHMARKS_DIR.glob("*.json")):
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

# Preserve order exactly as defined in JSON files (sorted by filename)
prompts = [item["prompt"] for item in mapping]

print("[PY] The number of prompts (problems):", len(prompts))
prompts

[PY] The number of prompts (problems): 11


['Build an arched bridge.',
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of red.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of blue.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of green.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of yellow.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of orange.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of purple.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of black.",
 "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of white.",
 'Lay the foundation for a 15x20 block rectangular building. Use stone blocks and make it two blocks deep.',
 'Lay the foundation for a 20x15 block rectangular building. Use stone blocks a

In [6]:
import subprocess, shutil, json
from pathlib import Path
from action_processor import read_coords_from_action

# Path to Node script in current directory
script = (Path.cwd() / "send_prompts.js").resolve()
node = shutil.which("node") or "node"

proc = subprocess.Popen(
    [node, str(script)],
    cwd=str(script.parent),       # Node's process.cwd() equals the JS folder
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,                    # line-buffered
)

# Send prompts to Node (single JSON line) and close stdin
proc.stdin.write(json.dumps({"prompts": prompts}) + "\n")
proc.stdin.close()

SENTINEL = "::ACTION_MAX_JS::" # indicator of the lastly executed JS file names for each prompt.
arr_of_coords = []

try:
    for line in proc.stdout:
        line = line.rstrip("\n")
        print(line)  # always mirror Node logs

        # If Node reports the max-numbered file, parse and print it
        if line.startswith(SENTINEL):
            payload_raw = line[len(SENTINEL):]
            try:
                payload = json.loads(payload_raw)
            except json.JSONDecodeError:
                print("[PY] Failed to parse sentinel JSON.")
                continue

            if payload.get("ok") and "path" in payload:
                file_path = Path(payload["path"])
                print(f"\n[PY] Max action file: index={payload.get('index')} name={payload.get('name')}")
                print(f"[PY] Path: {file_path}")

                try:
                    coords = read_coords_from_action(str(file_path))
                    print(f"[PY] len(coords): {len(coords)}")
                    arr_of_coords.append(coords)
                                       
                except Exception as e:
                    print(f"[PY] Failed to convert action to coords: {e}")
                    
            else:
                # e.g., dir not found or no js files
                reason = payload.get("reason", "unknown")
                print(f"[PY] No max file reported (reason={reason}).")
finally:
    proc.stdout.close()
    proc.wait()


🧹 Cleared all files under D:\git\mineCEraft\bots\andy\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=vlnR2SdrDifL-reZAAAD)

➡️ Sending to andy: "Build an arched bridge."
⏳ Waiting for completion keyword (timeout 20 min)...
📨 [andy] I'll build an arched bridge right where I am! !newAction("Build a small arched bridge using stone bricks, creating a curved arch over a small gap")
📨 [andy] I've built an arched stone brick bridge! It spans 11 blocks with a 3-block width and a nice curve in the middle. Added stone brick walls along the sides for safety too. How does it look? 
✅ Completion detected for "Build an arched bridge.".
::ACTION_MAX_JS::{"ok":true,"index":0,"name":"0.js","path":"D:\\git\\mineCEraft\\bots\\andy\\action-code\\0.js"}

[PY] Max action file: index=0 name=0.js
[PY] Path: D:\git\mineCEraft\bots\andy\action-code\0.js
[PY] len(coords): 61

➡️ Sending to andy: "Let's build a house with pillars made of stone, walls made of dirt, and a roof made of red

In [7]:
import json
import importlib
from pathlib import Path
from collections import defaultdict


# --- Configuration -----------------------------------------------------------
# The mapping file pairs each prompt with a list of checks (each check calls a function).
# Categories (e.g., material, size, planning, shape) are inferred from the first
# segment of each dotted function path, e.g., "size.is_equal" -> category "size".
BENCHMARKS_DIR = Path.cwd() / "benchmarks"

# --- Load the prompt -> checks mapping --------------------------------------
# Expected schema:
# [
#   {
#     "prompt": "...",
#     "checks": [
#       {"fn": "size.is_equal", "args": {"xz": [15, 20], "y": 2}},
#       ...
#     ]
#   },
#   ...
# ]
# Load all JSON files from benchmarks folder
mapping = []
for json_file in sorted(BENCHMARKS_DIR.glob("*.json")):
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

def resolve_callable(dotted: str):
    """
    Resolve 'size.is_equal' -> callable at 'eval_code.size.is_equal'.
    Keeps imports explicit and avoids eval().
    If the dotted path is not found, it will raise an ImportError.
    """
    fq = f"eval_code.{dotted}"
    mod_name, func_name = fq.rsplit(".", 1)
    mod = importlib.import_module(mod_name)
    return getattr(mod, func_name), fq

def run_checks(coords, checks):
    """
    Run checks on coords and return:
      - per-check results
      - aggregate score
      - per-category (module) subtotals
    """
    results = []
    score = 0
    cat_pass = defaultdict(int)   # category -> passed count
    cat_total = defaultdict(int)  # category -> total count

    for chk in checks:
        fn_name = chk["fn"]              # e.g., 'material.is_quantity_correct'
        args = chk.get("args") or {}
        category = fn_name.split(".", 1)[0]  # module name as category (e.g, material)

        fn, fq = resolve_callable(fn_name)
        try:
            ok = 1 if bool(fn(coords, **args)) else 0
        except Exception as e:
            ok, args = 0, {**args, "_error": str(e)}  # surface error on this line

        score += ok
        cat_total[category] += 1
        cat_pass[category]  += ok

        results.append({"fn": fq, "category": category, "ok": ok, "args": args})

    # shape as plain dicts for printing
    cat_summary = {
        c: {"pass": cat_pass[c], "total": cat_total[c]}
        for c in sorted(cat_total.keys())
    }
    return {"score": score, "total": len(results), "results": results, "by_category": cat_summary}

overall_pass = 0
overall_total = 0
overall_by_cat_pass  = defaultdict(int)
overall_by_cat_total = defaultdict(int)

for i in range(len(mapping)):
    spec   = mapping[i]
    prompt = spec["prompt"]
    checks = spec.get("checks", [])
    coords = arr_of_coords[i]

    report = run_checks(coords, checks)

    # --- Pretty print per-prompt report ---
    print("\n[PY] === Evaluation Result ===")
    print(f"[PY] Prompt #{i+1}: {prompt}")
    print(f"[PY] Score: {report['score']} / {report['total']} (coords={len(coords)})")

    # Category breakdown for this prompt
    print("[PY] Category scores:")
    for cat, st in report["by_category"].items():
        print(f"  - {cat}: {st['pass']} / {st['total']}")

    # Individual checks
    for r in report["results"]:
        status = "PASS" if r["ok"] else "FAIL"
        line = f"  · {status} | {r['fn']}({r.get('args', {})})"
        print(line)

    # Accumulate overall totals
    overall_pass  += report["score"]
    overall_total += report["total"]
    for cat, st in report["by_category"].items():
        overall_by_cat_pass[cat]  += st["pass"]
        overall_by_cat_total[cat] += st["total"]

# --- Overall summary across all prompts ---
if overall_total > 0:
    print("\n[PY] === Overall Summary ===")
    overall_pct = (overall_pass / overall_total * 100) if overall_total > 0 else 0.0
    print(f"[PY] Total PASS: {overall_pass} / {overall_total} ({overall_pct:.1f}%)")
    print("[PY] Category totals:")
    for cat in sorted(overall_by_cat_total.keys()):
        p = overall_by_cat_pass[cat]
        t = overall_by_cat_total[cat]
        cat_pct = (p / t * 100) if t > 0 else 0.0
        print(f"  - {cat}: {p} / {t} ({cat_pct:.1f}%)")



[PY] === Evaluation Result ===
[PY] Prompt #1: Build an arched bridge.
[PY] Score: 2 / 2 (coords=61)
[PY] Category scores:
  - shape: 1 / 1
  - structural_stability: 1 / 1
  · PASS | eval_code.structural_stability.is_ground_connected({})
  · PASS | eval_code.shape.center_column_above_center({})

[PY] === Evaluation Result ===
[PY] Prompt #2: Let's build a house with pillars made of stone, walls made of dirt, and a roof made of red.
[PY] Score: 8 / 8 (coords=300)
[PY] Category scores:
  - material: 7 / 7
  - structural_stability: 1 / 1
  · PASS | eval_code.structural_stability.is_ground_connected({})
  · PASS | eval_code.material.is_corner_material_equal_to({'expected_material': 'stone'})
  · PASS | eval_code.material.is_min_x_material_equal_to({'expected_material': 'dirt'})
  · PASS | eval_code.material.is_max_x_material_equal_to({'expected_material': 'dirt'})
  · PASS | eval_code.material.is_min_z_material_equal_to({'expected_material': 'dirt'})
  · PASS | eval_code.material.is_max_z

In [8]:
arr_of_coords[0]

[{'x': -5, 'y': 1, 'z': 0, 'material': 'stone_bricks'},
 {'x': -5, 'y': 0, 'z': 0, 'material': 'stone_bricks'},
 {'x': -5, 'y': 2, 'z': 0, 'material': 'stone_brick_wall'},
 {'x': -5, 'y': 1, 'z': 1, 'material': 'stone_bricks'},
 {'x': -5, 'y': 0, 'z': 1, 'material': 'stone_bricks'},
 {'x': -5, 'y': 1, 'z': 2, 'material': 'stone_bricks'},
 {'x': -5, 'y': 0, 'z': 2, 'material': 'stone_bricks'},
 {'x': -5, 'y': 2, 'z': 2, 'material': 'stone_brick_wall'},
 {'x': -4, 'y': 2, 'z': 0, 'material': 'stone_bricks'},
 {'x': -4, 'y': 3, 'z': 0, 'material': 'stone_brick_wall'},
 {'x': -4, 'y': 2, 'z': 1, 'material': 'stone_bricks'},
 {'x': -4, 'y': 2, 'z': 2, 'material': 'stone_bricks'},
 {'x': -4, 'y': 3, 'z': 2, 'material': 'stone_brick_wall'},
 {'x': -3, 'y': 3, 'z': 0, 'material': 'stone_bricks'},
 {'x': -3, 'y': 4, 'z': 0, 'material': 'stone_brick_wall'},
 {'x': -3, 'y': 3, 'z': 1, 'material': 'stone_bricks'},
 {'x': -3, 'y': 3, 'z': 2, 'material': 'stone_bricks'},
 {'x': -3, 'y': 4, 'z': 2, '

In [9]:
from eval_code.viz import plotly_blocks

plotly_blocks.plot(arr_of_coords[0])

### Remove Agent

In [10]:
proc_agent.kill()
print("Process terminated.")

Process terminated.
